# STCIR — Semi-Synthetic Test Collection Framework

**Pipeline overview:**
1. Corpus acquisition & chunking
2. Topic creation (domain mode) or loading (standard mode)
3. Candidate pool construction (BM25 + bi-encoders → RRF Stage-1 → top-1000)
4. Cross-encoder reranking (5–6 models → RRF Stage-2 → top-10)
5. Relevance annotation (human Flask UI or Gemma4 LLM)
6. IR metrics evaluation
7. System-level correlation (Kendall's τ, Spearman's ρ)

**Supported languages:** `arabic` | `english`  
**Supported datasets:** `mrtydi_arabic` · `mmarco_arabic` · `algerian_legal` · `mrtydi_english` · `msmarco`

In [ ]:
# Install the package (first run only)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
print("✅ stcir installed")

## Cell 0 — Configuration

In [ ]:
from stcir import STCIRConfig
from stcir.registry import MODEL_REGISTRY
from stcir.utils import Checkpoint, get_logger
import logging
logging.basicConfig(level=logging.INFO)

# ═══════════════════════════════════════════════════════════════════════════
# CHOOSE LANGUAGE
# ───────────────────────────────────────────────────────────────────────────
LANGUAGE = "arabic"      # "arabic"  |  "english"

# ═══════════════════════════════════════════════════════════════════════════
# CHOOSE DATASET
# arabic  → "mrtydi_arabic" | "mmarco_arabic" | "algerian_legal"
# english → "mrtydi_english" | "msmarco"      | "custom_domain"
# ───────────────────────────────────────────────────────────────────────────
DATASET = "mrtydi_arabic"

# ═══════════════════════════════════════════════════════════════════════════
# OR load from a YAML config file (overrides the fields above)
# config = STCIRConfig.from_yaml("configs/mrtydi_arabic.yaml")
# ───────────────────────────────────────────────────────────────────────────

config = STCIRConfig(
    language = LANGUAGE,
    dataset  = DATASET,

    # ── Mode ────────────────────────────────────────────────────────────
    # "standard" = use existing corpus + topics from a benchmark
    # "domain"   = chunk raw corpus + create topics from scratch
    mode = "standard",

    # ── Corpus (domain mode only) ────────────────────────────────────────
    # corpus_path = "data/algerian_legal_corpus.jsonl",

    # ── Stage-1 source ───────────────────────────────────────────────────
    # "computed"         → run BM25 + bi-encoders in this session
    # "prebuilt_mrtydi"  → download pre-built runs from HuggingFace
    # "prebuilt_mmarco"  → download pre-built runs from HuggingFace
    stage1_source = "prebuilt_mrtydi",

    # ── Stage-2 source ───────────────────────────────────────────────────
    # "computed"         → run cross-encoders locally
    # "prebuilt_mrtydi"  → download MrTydi_second_stage from HuggingFace
    stage2_source = "prebuilt_mrtydi",

    # ── Chunking (domain mode only) ──────────────────────────────────────
    chunk_max_tokens = 512,
    chunk_stride     = 50,
    chunk_min_tokens = 20,

    # ── Topics (domain mode only) ────────────────────────────────────────
    n_topics    = 100,
    topic_mode  = "human",   # "human" | "llm"
    topic_seed  = 42,

    # ── Retrieval ────────────────────────────────────────────────────────
    bm25_top_k  = 1000,
    dense_top_k = 1000,
    pool_top_k  = 1000,
    rrf_k       = 60,
    final_top_k = 10,

    # ── Annotation ───────────────────────────────────────────────────────
    annotation_mode = "human",   # "human" | "llm"
    llm_model       = "google/gemma-3-4b-it",

    # ── Evaluation ───────────────────────────────────────────────────────
    metrics = ["MRR@10", "nDCG@10", "MAP", "Recall@10", "Hit@10", "P@10"],
    # Path to human qrels for correlation analysis (optional)
    # qrels_reference = "data/qrels.test.mrtydi.txt",

    # ── Hardware ─────────────────────────────────────────────────────────
    device            = "cuda",  # "cuda" | "cpu"
    encode_batch_size = 64,
    rerank_batch_size = 32,
)

# Checkpointing — resume interrupted runs
ckpt = Checkpoint(config.cache_path("checkpoints"))

print(f"Language  : {config.language}")
print(f"Dataset   : {config.dataset}")
print(f"Mode      : {config.mode}")
print(f"Stage-1   : {config.stage1_source}")
print(f"Stage-2   : {config.stage2_source}")
print(f"HF repo   : {config.prebuilt_hf_repo}")
print(f"Bi-encoders ({len(config.bi_encoders)}): {[m.split('/')[-1] for m in config.bi_encoders]}")
print(f"Cross-encoders ({len(config.cross_encoders)}): {[m.split('/')[-1] for m in config.cross_encoders]}")


## Cell 1 — Phase 1: Corpus Acquisition & Chunking

In [ ]:
import pandas as pd
from stcir.corpus.loader import load_corpus
from stcir.corpus.chunker import TokenAwareChunker
from stcir.topics.loader import load_corpus_ir
from stcir.utils import save_jsonl, load_jsonl

corpus_cache = config.cache_path("corpus", "passages.jsonl")

if ckpt.is_done("phase1"):
    print("⏩ Phase 1 already done — loading from cache")
    passages = load_jsonl(corpus_cache)

elif config.mode == "domain":
    # ── Domain mode: load raw corpus and chunk it ─────────────────────────
    print("📄 Loading raw corpus from:", config.corpus_path)
    raw_passages = load_corpus(config.corpus_path)
    chunker = TokenAwareChunker(
        tokenizer_name = config.bi_encoders[0],
        max_tokens     = config.chunk_max_tokens,
        stride         = config.chunk_stride,
        min_tokens     = config.chunk_min_tokens,
    )
    passages = chunker.chunk_passages(raw_passages)
    save_jsonl(passages, corpus_cache)
    ckpt.mark_done("phase1")

else:
    # ── Standard mode: load benchmark corpus ─────────────────────────────
    # Primary  : HuggingFace (hatemestinbejaia/mr-tydi-ar-dev or mmarco-subset)
    # Fallback : ir_datasets (mr-tydi/ar, mmarco/v2/ar, msmarco-passage, …)
    print(f"📚 Loading corpus for '{config.dataset}' (HF primary / ir_datasets fallback) …")
    passages = load_corpus_ir(config.dataset)
    save_jsonl(passages, corpus_cache)
    ckpt.mark_done("phase1")

# ── Summary ──────────────────────────────────────────────────────────────
lengths = [len(p["text"].split()) for p in passages[:5000]]
print(f"\n✅ Corpus ready: {len(passages):,} passages")
print(f"   Avg length : {sum(lengths)/len(lengths):.1f} words (sample of {len(lengths)})")
print(f"   Min / Max  : {min(lengths)} / {max(lengths)} words")

# Build a fast pid → text lookup (used in Cells 6-8)
passage_lookup = {p["pid"]: p["text"] for p in passages}
print(f"\nPassage lookup built: {len(passage_lookup):,} entries")


## Cell 2 — Phase 2: Topic Creation / Loading

In [ ]:
import threading
from stcir.topics.loader import load_topics, load_topics_from_file
from stcir.topics.selector import sample_passages_for_topics, save_topic_seeds
from stcir.utils import save_jsonl

topics_cache = config.cache_path("topics", "topics.jsonl")

if ckpt.is_done("phase2"):
    print("⏩ Phase 2 already done — loading topics from cache")
    topics = load_topics_from_file(topics_cache)

elif config.mode == "standard":
    # ── Standard: load benchmark queries ─────────────────────────────────
    # Primary  : HuggingFace (hatemestinbejaia/mr-tydi-ar-dev or mmarco-subset)
    # Fallback : ir_datasets ("test" split for Mr. TyDi, "dev" for mMARCO/MS MARCO)
    print(f"📋 Loading topics for '{config.dataset}' (HF primary / ir_datasets fallback) …")
    topics = load_topics(config.dataset)   # uses HF primary; falls back to ir_datasets
    save_jsonl(topics, topics_cache)
    ckpt.mark_done("phase2")

elif config.mode == "domain":
    seeds_path  = config.cache_path("topics", "seeds.jsonl")
    output_path = config.cache_path("topics", "topics.jsonl")

    if not seeds_path.exists():
        seeds = sample_passages_for_topics(passages, n=config.n_topics, seed=config.topic_seed)
        save_topic_seeds(seeds, seeds_path)
        print(f"🌱 {len(seeds)} passage seeds saved → {seeds_path}")

    if config.topic_mode == "human":
        from stcir.ui.topic_ui import run_topic_ui
        print(f"\n🌐 Starting Topic Creation UI on port {config.flask_port + 1} ...")
        print(f"   Open: http://localhost:{config.flask_port + 1}")
        t = threading.Thread(
            target=run_topic_ui,
            kwargs=dict(
                seeds_path  = str(seeds_path),
                output_path = str(output_path),
                host        = config.flask_host,
                port        = config.flask_port + 1,
                lang        = config.language,
            ),
            daemon=True,
        )
        t.start()
        print("⚠️  Re-run this cell after finishing annotation to continue.")
        topics = []

    else:  # LLM topic generation
        from stcir.annotation.topic_gen import GemmaTopicGenerator
        from stcir.utils import load_jsonl as _lj
        seeds = _lj(seeds_path)
        gen = GemmaTopicGenerator(
            model_name = config.llm_model,
            device     = config.device,
            language   = config.language,
        )
        topics = gen.generate(seeds)
        save_jsonl(topics, output_path)
        ckpt.mark_done("phase2")

    if output_path.exists():
        topics = load_topics_from_file(output_path)
        if len(topics) >= config.n_topics:
            ckpt.mark_done("phase2")

if topics:
    print(f"\n✅ Topics ready: {len(topics)} queries")
    import pandas as pd
    display(pd.DataFrame([{"qid": t["qid"], "query": t["text"]} for t in topics[:5]]))


## Cell 3 — Phase 3a: BM25 Indexing + Retrieval

> **Skipped** when `stage1_source != 'computed'`

In [ ]:
from stcir.indexing.bm25 import BM25Index
from stcir.retrieval.bm25_retriever import BM25Retriever
from stcir.utils import save_run, load_run, Timer

bm25_run_path = config.cache_path("runs", "bm25.tsv")
bm25_run = {}

if config.stage1_source != "computed":
    print(f"⏭  Phase 3a SKIPPED — stage1_source='{config.stage1_source}' (will use pre-built runs)")

elif ckpt.is_done("phase3a"):
    print("⏩ Phase 3a already done — loading BM25 run from cache")
    bm25_run = load_run(bm25_run_path)

else:
    bm25_index_path = config.cache_path("indexes", "bm25.pkl")

    if bm25_index_path.exists():
        print("⏩ BM25 index exists — loading ...")
        bm25_idx = BM25Index.load(bm25_index_path)
    else:
        with Timer("BM25 index build"):
            bm25_idx = BM25Index().build(passages)
        bm25_idx.save(bm25_index_path)

    retriever = BM25Retriever(bm25_idx)
    with Timer("BM25 retrieval"):
        bm25_run = retriever.retrieve(topics, top_k=config.bm25_top_k)

    save_run(bm25_run, bm25_run_path)
    ckpt.mark_done("phase3a")

    print(f"\n✅ BM25 run: {len(bm25_run)} topics, top-{config.bm25_top_k} per topic")

## Cell 4 — Phase 3b: Bi-Encoder Encoding + FAISS Dense Retrieval

> **Skipped** when `stage1_source != 'computed'`

In [ ]:
import numpy as np
from stcir.indexing.encoder import BiEncoder
from stcir.indexing.faiss_index import FaissIndex
from stcir.retrieval.dense_retriever import DenseRetriever
from stcir.utils import save_run, load_run, Timer

dense_runs   = []   # will be filled below
dense_labels = []

if config.stage1_source != "computed":
    print(f"⏭  Phase 3b SKIPPED — stage1_source='{config.stage1_source}'")

else:
    for model_name in config.bi_encoders:
        short = model_name.split("/")[-1]
        run_path     = config.cache_path("runs", f"dense_{short}.tsv")
        index_path   = config.cache_path("indexes", f"{short}.faiss")
        pids_path    = config.cache_path("indexes", f"{short}.pids.pkl")
        corpus_emb_path = config.cache_path("embeddings", f"{short}.npy")

        if run_path.exists():
            print(f"⏩ {short}: run exists — loading")
            dense_runs.append(load_run(run_path))
            dense_labels.append(short)
            continue

        print(f"\n🔷 Processing bi-encoder: {short}")
        encoder = BiEncoder(model_name, device=config.device)

        # Encode corpus
        if corpus_emb_path.exists():
            print(f"   Loading cached corpus embeddings ...")
            corpus_embs = np.load(corpus_emb_path)
        else:
            with Timer(f"  Encoding corpus [{short}]"):
                corpus_embs = encoder.encode_corpus(passages, batch_size=config.encode_batch_size)
            corpus_emb_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(corpus_emb_path, corpus_embs)

        # Build / load FAISS index
        if index_path.exists():
            faiss_idx = FaissIndex.load(index_path, pids_path)
        else:
            pids = [p["pid"] for p in passages]
            with Timer(f"  Building FAISS index [{short}]"):
                faiss_idx = FaissIndex().build(corpus_embs, pids)
            faiss_idx.save(index_path, pids_path)

        # Dense retrieval
        retriever = DenseRetriever(encoder, faiss_idx)
        with Timer(f"  Dense retrieval [{short}]"):
            run = retriever.retrieve(topics, top_k=config.dense_top_k,
                                     batch_size=config.encode_batch_size)

        save_run(run, run_path)
        dense_runs.append(run)
        dense_labels.append(short)
        print(f"   ✅ {short}: {len(run)} topics, top-{config.dense_top_k}")

    if not ckpt.is_done("phase3b"):
        ckpt.mark_done("phase3b")

    print(f"\n✅ Phase 3b done: {len(dense_runs)} bi-encoder runs")

## Cell 5 — Phase 3c: RRF Stage-1 → Candidate Pool of 1000

In [ ]:
from stcir.retrieval.rrf import Stage1Pool
from stcir.registry import PREBUILT_FOLDER_MAP
from stcir.utils import save_run, load_run

pool_path = config.cache_path("runs", "stage1_pool.tsv")

if ckpt.is_done("phase3c"):
    print("⏩ Phase 3c already done — loading pool from cache")
    pool = load_run(pool_path)

else:
    stage1 = Stage1Pool(rrf_k=config.rrf_k, top_k=config.pool_top_k)

    if config.stage1_source == "computed":
        assert bm25_run,   "BM25 run is empty — re-run Cell 3"
        assert dense_runs, "Dense runs are empty — re-run Cell 4"
        print(f"🔀 RRF Stage-1: fusing BM25 + {len(dense_runs)} dense runs ...")
        pool = stage1.from_computed(
            bm25_run   = bm25_run,
            dense_runs = dense_runs,
            run_labels = dense_labels,
        )
    else:
        hf_folder = PREBUILT_FOLDER_MAP["stage1"][config.stage1_source]
        print(f"⬇️  Downloading pre-built Stage-1: {config.prebuilt_hf_repo}/{hf_folder} ...")
        pool = stage1.from_prebuilt(
            hf_repo   = config.prebuilt_hf_repo,
            hf_folder = hf_folder,
            cache_dir = str(config.cache_path("prebuilt")),
        )

    save_run(pool, pool_path)
    ckpt.mark_done("phase3c")

pool_sizes = [len(v) for v in pool.values()]
print(f"\n✅ Stage-1 pool: {len(pool)} topics")
print(f"   Avg pool size : {sum(pool_sizes)/len(pool_sizes):.0f} candidates/topic")
print(f"   Min / Max     : {min(pool_sizes)} / {max(pool_sizes)}")


## Cell 6 — Phase 4: Cross-Encoder Reranking → RRF Stage-2 → Top-10

In [ ]:
from stcir.reranking.cross_encoder import CrossEncoderReranker
from stcir.reranking.rrf_rerank import rerank_with_rrf
from stcir.retrieval.rrf import Stage1Pool
from stcir.registry import PREBUILT_FOLDER_MAP
from stcir.utils import save_run, load_run, Timer

top10_run_path = config.cache_path("runs", "top10_reranked.tsv")

if ckpt.is_done("phase4"):
    print("⏩ Phase 4 already done — loading top-10 run from cache")
    top10_run = load_run(top10_run_path)

elif config.stage2_source != "computed":
    hf_folder = PREBUILT_FOLDER_MAP["stage2"][config.stage2_source]
    print(f"⬇️  Downloading pre-built Stage-2: {config.prebuilt_hf_repo}/{hf_folder} ...")
    top10_run = Stage1Pool(rrf_k=config.rrf_k, top_k=config.final_top_k).from_prebuilt(
        hf_repo   = config.prebuilt_hf_repo,
        hf_folder = hf_folder,
        cache_dir = str(config.cache_path("prebuilt")),
    )
    save_run(top10_run, top10_run_path)
    ckpt.mark_done("phase4")

else:
    ce_runs, ce_labels = [], []

    for model_name in config.cross_encoders:
        short    = model_name.split("/")[-1]
        run_path = config.cache_path("runs", f"ce_{short}.tsv")

        if run_path.exists():
            print(f"⏩ {short}: run exists — loading")
            ce_runs.append(load_run(run_path))
            ce_labels.append(short)
            continue

        print(f"\n🔶 Cross-encoder: {short}")
        reranker = CrossEncoderReranker(model_name, device=config.device)
        with Timer(f"  Reranking [{short}]"):
            run = reranker.rerank(
                topics         = topics,
                pool           = pool,
                passage_lookup = passage_lookup,
                top_k          = config.rerank_top_k,
                batch_size     = config.rerank_batch_size,
            )
        save_run(run, run_path)
        ce_runs.append(run)
        ce_labels.append(short)
        print(f"   ✅ {short} done")

    print(f"\n🔀 RRF Stage-2: fusing {len(ce_runs)} cross-encoder runs → top {config.final_top_k}")
    top10_run = rerank_with_rrf(
        cross_encoder_runs = ce_runs,
        run_labels         = ce_labels,
        rrf_k              = config.rrf_k,
        top_k              = config.final_top_k,
    )
    save_run(top10_run, top10_run_path)
    ckpt.mark_done("phase4")

sample_qid = next(iter(top10_run))
print(f"\n✅ Top-10 run ready: {len(top10_run)} topics")
print(f"\nSample (qid={sample_qid}):")
import pandas as pd
display(pd.DataFrame(
    [(r+1, pid, f"{sc:.4f}", passage_lookup.get(pid,"")[:80]+"...")
     for r,(pid,sc) in enumerate(top10_run[sample_qid])],
    columns=["rank","pid","score","passage_preview"]
))


## Cell 7 — Phase 5: Relevance Annotation (Human Flask UI or Gemma4 LLM)

In [ ]:
import threading
from stcir.utils import save_qrels, load_qrels

qrels_path   = config.output_path("qrels.tsv")
top10_ui_path = config.cache_path("annotation", "top10_for_ui.jsonl")

if ckpt.is_done("phase5"):
    print("⏩ Phase 5 already done — loading qrels from cache")
    qrels = load_qrels(qrels_path)

elif config.annotation_mode == "llm":
    # ── LLM Annotation (Gemma4) ──────────────────────────────────────────
    from stcir.annotation.llm import GemmaAnnotator
    print(f"🤖 LLM annotation with {config.llm_model} ...")
    annotator = GemmaAnnotator(
        model_name = config.llm_model,
        device     = config.device,
        language   = config.language,
    )
    qrels = annotator.annotate(
        topics         = topics,
        top10_run      = top10_run,
        passage_lookup = passage_lookup,
        batch_size     = config.llm_batch_size,
    )
    save_qrels(qrels, qrels_path)
    ckpt.mark_done("phase5")

else:
    # ── Human Annotation (Flask UI) ──────────────────────────────────────
    from stcir.ui.relevance_ui import prepare_top10_for_ui, run_relevance_ui

    # Prepare JSONL file for the UI
    prepare_top10_for_ui(
        top10_run      = top10_run,
        topics         = topics,
        passage_lookup = passage_lookup,
        output_path    = str(top10_ui_path),
    )

    print(f"\n🌐 Starting Relevance Annotation UI on port {config.flask_port} ...")
    print(f"   Open: http://localhost:{config.flask_port}")
    print("   Annotate all topics, then re-run this cell to load results.")

    t = threading.Thread(
        target=run_relevance_ui,
        kwargs=dict(
            top10_path  = str(top10_ui_path),
            output_path = str(qrels_path),
            host        = config.flask_host,
            port        = config.flask_port,
            lang        = config.language,
        ),
        daemon=True,
    )
    t.start()

    # Check if already partially/fully annotated
    if qrels_path.exists():
        qrels = load_qrels(qrels_path)
        print(f"   Current progress: {len(qrels)}/{len(top10_run)} topics annotated")
        if len(qrels) == len(top10_run):
            ckpt.mark_done("phase5")
    else:
        qrels = {}
        print("   ⚠️  Re-run this cell after completing annotation.")

if qrels:
    n_rel = sum(v for d in qrels.values() for v in d.values())
    print(f"\n✅ Qrels ready: {len(qrels)} topics, {n_rel} relevant judgments")
    print(f"   Saved → {qrels_path}")

## Cell 8 — Evaluation: IR Metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from stcir.evaluation.metrics import evaluate_multiple_runs, hit_at_k
from stcir.utils import load_run
import glob, os

assert qrels, "Qrels are empty — complete Phase 5 first"

# ── Collect all individual system runs ───────────────────────────────────
runs_dir    = config.cache_path("runs")
system_runs = {}

# BM25
bm25_path = runs_dir / "bm25.tsv"
if bm25_path.exists():
    system_runs["BM25"] = load_run(bm25_path)

# Dense retrievers
for model_name in config.bi_encoders:
    short = model_name.split("/")[-1]
    p = runs_dir / f"dense_{short}.tsv"
    if p.exists():
        system_runs[f"Dense-{short}"] = load_run(p)

# Stage-1 pool (RRF)
pool_path = runs_dir / "stage1_pool.tsv"
if pool_path.exists():
    system_runs["Pool-RRF"] = load_run(pool_path)

# Cross-encoder re-rankers
for model_name in config.cross_encoders:
    short = model_name.split("/")[-1]
    p = runs_dir / f"ce_{short}.tsv"
    if p.exists():
        system_runs[f"CE-{short}"] = load_run(p)

# Full pipeline (top-10)
top10_path = runs_dir / "top10_reranked.tsv"
if top10_path.exists():
    system_runs["Full-Pipeline"] = load_run(top10_path)

print(f"Evaluating {len(system_runs)} systems ...")

# ── Evaluate ─────────────────────────────────────────────────────────────
results_df = evaluate_multiple_runs(qrels, system_runs, config.metrics)

# Add Hit@10 manually
results_df["Hit@10"] = pd.Series({
    name: hit_at_k(qrels, run, k=10)
    for name, run in system_runs.items()
})

# Save
results_path = config.output_path("evaluation_results.csv")
results_df.to_csv(results_path)
print(f"\nSaved → {results_path}")

# ── Display ──────────────────────────────────────────────────────────────
display(results_df.style.highlight_max(axis=0, color="#c8e6c9").format("{:.4f}"))

# ── Bar chart ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
results_df[["MRR@10", "nDCG@10", "Recall@10"]].plot(kind="bar", ax=ax)
ax.set_title(f"System comparison — {config.dataset}")
ax.set_ylabel("Score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout()
plt.savefig(config.output_path("evaluation_barplot.png"), dpi=150)
plt.show()

## Cell 9 — Evaluation: System-Level Correlation (Kendall's τ · Spearman's ρ)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from stcir.evaluation.correlation import (
    compute_system_correlation,
    global_rank_correlation,
    plot_system_scatter,
)

# ── Option A: compare synthetic qrels vs human reference ─────────────────────
# Set config.qrels_reference to a local TREC qrels file  OR  set
# use_hf_qrels = True to load them from HuggingFace primary (ir_datasets fallback).
use_hf_qrels = True   # ← flip to False if you have a local file

if config.qrels_reference:
    from stcir.utils import load_qrels as _lq
    human_qrels = _lq(config.qrels_reference)
    have_human  = True

elif use_hf_qrels and config.dataset in ("mrtydi_arabic", "mrtydi_english",
                                          "mmarco_arabic", "msmarco"):
    from stcir.topics.loader import load_reference_qrels
    # Primary  : HuggingFace (hatemestinbejaia/mr-tydi-ar-dev or mmarco-subset)
    # Fallback : ir_datasets
    print(f"📥 Loading reference qrels (HF primary / ir_datasets fallback) for {config.dataset} …")
    human_qrels = load_reference_qrels(config.dataset)
    have_human  = True

else:
    have_human = False

if have_human:
    print("📊 System-level correlation (synthetic vs. human qrels) ...")
    corr_df = compute_system_correlation(
        human_qrels     = human_qrels,
        synthetic_qrels = qrels,
        runs            = system_runs,
        metrics         = ["MRR@10", "nDCG@10", "Recall@10"],
    )
    display(corr_df.style.format("{:.4f}"))
    corr_df.to_csv(config.output_path("system_correlation.csv"))

    # Global pair-level correlation
    import tempfile, os
    from stcir.utils import save_qrels
    tmp = tempfile.NamedTemporaryFile(suffix=".tsv", delete=False)
    tmp.close()
    save_qrels(human_qrels,    tmp.name + ".human")
    save_qrels(qrels,          tmp.name + ".synth")
    global_corr = global_rank_correlation(
        human_qrels_path     = tmp.name + ".human",
        synthetic_qrels_path = tmp.name + ".synth",
    )
    os.unlink(tmp.name + ".human")
    os.unlink(tmp.name + ".synth")

    print(f"\nGlobal Kendall's τ  : {global_corr['global_kendall_tau']:.4f}")
    print(f"Global Spearman's ρ : {global_corr['global_spearman_rho']:.4f}")

    for metric in ["MRR@10", "nDCG@10", "Recall@10"]:
        fig = plot_system_scatter(
            human_qrels     = human_qrels,
            synthetic_qrels = qrels,
            runs            = system_runs,
            metric          = metric,
            save_path       = str(config.output_path(f"scatter_{metric.replace('@','_')}.png")),
        )
        plt.show()

else:
    # ── Option B: self-consistency ranking (no human reference) ──────────
    print("ℹ️  No human reference. Computing cross-metric rank agreement.")
    ranking_by_metric = {col: results_df[col].rank(ascending=False)
                         for col in results_df.columns}
    rank_df = pd.DataFrame(ranking_by_metric)
    display(rank_df.style.format("{:.0f}"))

    from scipy.stats import kendalltau, spearmanr
    cols = [c for c in rank_df.columns if "@" in c]
    if len(cols) >= 2:
        tau, _ = kendalltau(rank_df[cols[0]], rank_df[cols[1]])
        rho, _ = spearmanr(rank_df[cols[0]], rank_df[cols[1]])
        print(f"\nRank agreement {cols[0]} vs {cols[1]}:")
        print(f"  Kendall's τ  : {tau:.4f}")
        print(f"  Spearman's ρ : {rho:.4f}")

print("\n✅ Evaluation complete. All outputs saved to:", config.output_dir)
